In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from itertools import combinations
from note.v2.rule_extractor import measure_rules_2
import sympy

from box import Box
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from mlflow import MlflowClient
from sklearn.tree import DecisionTreeClassifier
from note.v2.rule_extractor import measure_rules

from note.v2.rule_extractor import get_rules
from note.note import run, DEFAULT_PARAMS
from datasetz.core.load_dataset import load_embedded_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from toolz.curried import pipe, filter, map, reduce

import joblib
import networkit as nk
import networkx as nx
import igraph as ig

In [3]:
meta_prams = {"n_jobs": 8, "a": "bcc"}

In [6]:
"," + ",".join([f"{k}={v}" for k, v in meta_prams.items()])

',n_jobs=8,a=bcc'

In [3]:
dataset = load_embedded_dataset('keel-embedded', 'wine')

splitter = StratifiedKFold()
train_test_dataset = dataset \
    .encode_x_to_labels() \
    .encode_y_to_numeric_labels() \
    .train_test_split(splitter)[0]

/Users/bgulowaty/studia/projekty/datasets/definitions
{PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-embedded-splits.yml'), PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-splits.yml'), PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-embedded.yml')}
________________________________________________________________________________
[Memory] Calling datasetz.core.load_dataset.load...
load()
_____________________________________________________________load - 0.0s, 0.0min
________________________________________________________________________________
[Memory] Calling datasetz.api.dataset.x...
x()
________________________________________________________________x - 0.0s, 0.0min
________________________________________________________________________________
[Memory] Calling datasetz.api.dataset.y...
y()
________________________________________________________________y - 0.0s, 0.0min
___________________________________________

In [7]:
dt = DecisionTreeClassifier()
dt.fit(train_test_dataset.train.x, train_test_dataset.train.y)

DecisionTreeClassifier()

In [3]:
from sympy import Symbol
from sympy import solve_rational_inequalities, Poly, reduce_inequalities
from sympy.parsing.sympy_parser import parse_expr

In [4]:
dataset = load_embedded_dataset('keel-embedded', "thyroid")

splitter = StratifiedKFold()
train_test_dataset = dataset \
    .encode_x_to_labels() \
    .encode_y_to_numeric_labels() \
    .train_test_split(splitter)[0]

rf = RandomForestClassifier(random_state=42, n_estimators=10)
rf.fit(train_test_dataset.train.x, train_test_dataset.train.y)

# results = run(train_test_dataset.train.x, train_test_dataset.train.y, rf, Box(DEFAULT_PARAMS))

/Users/bgulowaty/studia/projekty/datasets/definitions
{PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-splits.yml'), PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-embedded.yml'), PosixPath('/Users/bgulowaty/studia/projekty/datasets/definitions/keel-embedded-splits.yml')}
________________________________________________________________________________
[Memory] Calling datasetz.core.load_dataset.DatasetDefinition.load...
load()
_____________________________________________________________load - 0.1s, 0.0min
________________________________________________________________________________
[Memory] Calling datasetz.api.dataset.SimpleDataset.x...
x()
________________________________________________________________x - 0.0s, 0.0min
________________________________________________________________________________
[Memory] Calling datasetz.api.dataset.SimpleDataset.y...
y()
________________________________________________________________y - 0.0s, 0.0m

RandomForestClassifier(n_estimators=10, random_state=42)

In [6]:
import scipy

In [8]:


all_rules = pipe(
    rf.estimators_,
    map(lambda estimator: get_rules(estimator)),
    reduce(tuple.__add__),
    set
    # map(lambda r: join_consecutive_statements(r)),
)

In [46]:
rules_as_inequalities = [
    reduce_inequalities([parse_expr(f"f{statement[0]} {statement[1]} {statement[2]}") for statement in rule]) for rule in all_rules
]

In [14]:
from sympy.printing.aesaracode import aesara_function

In [15]:
aesara_function(reduce_inequalities)

TypeError: aesara_function() missing 1 required positional argument: 'outputs'

In [13]:
ineq = reduce_inequalities([parse_expr("f20 < 3"), parse_expr("f15 >5")])

In [11]:
ineq

(-oo < f20) & (5 < f15) & (f15 < oo) & (f20 < 3)

In [79]:
list([str(s) for s in ineq.free_symbols])

['f20', 'f15']

In [82]:
sorted({"f15", "f1"}, key=lambda it: int(it[1:]))

['f1', 'f15']

In [102]:
f = sympy.lambdify(["f20", "f15", 'f10'],ineq, modules=['numpy'])

In [98]:
sample = np.array([
    [1,2,3,4],
    [1,2,3,4]
          ])

In [108]:
f(1,2,3)

False

In [119]:
np.any(np.full(23, False, dtype=object) == True)

False

In [113]:
np.apply_along_axis(lambda row: f(*row), 1, sample[:, :a3])

array([False, False])

In [121]:
sample[[False, True]] = [0,0,0,0]

In [122]:
sample

array([[1, 2, 3, 4],
       [0, 0, 0, 0]])

In [105]:
f(sample[:, :3])

TypeError: _lambdifygenerated() missing 2 required positional arguments: 'f15' and 'f10'

In [97]:
print(*sample[:3])

1 2 3


In [96]:
f(*sample[:3])

False

In [47]:
len(set(rules_as_inequalities))

1250

In [11]:
all_rules

TypeError: 'set' object is not subscriptable

In [31]:
reduce_inequalities(parse_expr("f1 > 3")).as_set()

Interval.open(3, oo)

In [32]:
reduce_inequalities([parse_expr("f1 > 3"), parse_expr("f1 > 15")]).as_set()

Interval.open(15, oo)

In [43]:
len({
    reduce_inequalities([parse_expr("f1 > 15"), parse_expr("f2 < 3")]),
    reduce_inequalities([parse_expr("f1 > 3"), parse_expr("f1 > 15"), parse_expr("f2 < 3")])
})

1

In [39]:
reduce_inequalities([parse_expr("f1 > 3"), parse_expr("f1 > 15"), parse_expr("f2 < 3")]) == reduce_inequalities([parse_expr("f1 > 15"), parse_expr("f2 < 3")])

True

In [288]:
rule_measurements_2 = measure_rules_2(all_rules, n_jobs=8)

Generated 780625 combinations


measure_rules:  56%|█████▌    | 437328/780625 [47:40<24:39, 232.11it/s]  Exception ignored in: <function tqdm.__del__ at 0x28314d990>
Traceback (most recent call last):
  File "/Users/bgulowaty/Library/Caches/pypoetry/virtualenvs/non-overlapping-rules-ensemble-ztoKC6G--py3.10/lib/python3.10/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Users/bgulowaty/Library/Caches/pypoetry/virtualenvs/non-overlapping-rules-ensemble-ztoKC6G--py3.10/lib/python3.10/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'
measure_rules: 100%|██████████| 780625/780625 [1:28:14<00:00, 147.44it/s]  


In [248]:
import tqdm

In [253]:
combs = list(combinations(all_rules_as_sets, 2))

In [271]:
all_rules_as_sets[combs[:3][0][0]]

{0: [Interval(-oo, 53.0000000000000), Interval.open(50.5000000000000, oo)],
 1: [Interval.open(0.500000000000000, oo)],
 7: [Interval(-oo, 0.500000000000000)],
 9: [Interval(-oo, 0.500000000000000)],
 15: [Interval(-oo, 0.500000000000000)],
 16: [Interval.open(164.500000000000, oo)],
 17: [Interval.open(13.5000000000000, oo),
  Interval.open(23.5000000000000, oo)],
 19: [Interval.open(33.5000000000000, oo)],
 20: [Interval.open(80.5000000000000, oo),
  Interval(-oo, 160.500000000000),
  Interval(-oo, 91.0000000000000)]}

In [290]:
np.unique(list(dict(rule_measurements_2).values()), return_counts=True)

(array([False,  True]), array([632085, 148540]))

In [289]:
np.unique(list(rule_measurements.values()), return_counts=True)

(array([False,  True]), array([632085, 148540]))

In [298]:
inequalities_combinations = list(combinations(simplified_inequalities, 2))

In [299]:
with tqdm_joblib(tqdm(desc="measure_rules", total=len(inequalities_combinations))) as progress_bar:
    measured_combinations_3 = Parallel(n_jobs=8, backend='loky')(
        delayed(
            lambda c: (c, reduce_inequalities(c[0], c[1]) != False)
        )(combination)
        for combination in inequalities_combinations
    )

measure_rules: 100%|██████████| 780625/780625 [23:51<00:00, 545.47it/s]  


In [292]:
%%timeit
all_inequalities_combinations = list(combinations(all_rules_as_inequalities, 2))
with tqdm_joblib(tqdm(desc="measure_rules", total=len(all_inequalities_combinations))) as progress_bar:
    measured_combinations_2 = Parallel(n_jobs=8, backend='loky')(
        delayed(
            lambda c: (c, reduce_inequalities(c[0] + c[1]) != False)
        )(combination)
        for combination in all_inequalities_combinations
    )

measure_rules:  20%|█▉        | 154972/780625 [15:56<1:04:22, 161.99it/s]


KeyboardInterrupt: 

In [264]:
all_rules_as_inequalities[0]

[f20 > 86.5,
 f17 <= 23.5,
 f20 > 232.5,
 f20 <= 297.5,
 f19 <= 83.5,
 f0 <= 81.5,
 f2 <= 0.5,
 f17 <= 21.5,
 f18 > 93.5,
 f15 > 0.5,
 f0 <= 62.0]

In [261]:
reduce_inequalities(all_rules_as_inequalities[0])

(-oo < f2) & (-oo < f19) & (-oo < f17) & (-oo < f0) & (f0 <= 62.0) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 297.5) & (0.5 < f15) & (93.5 < f18) & (232.5 < f20) & (f15 < oo) & (f18 < oo)

In [250]:
simplified_inequalities[0]

(f0 <= 62.0) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 297.5) & (f15 > 0.5) & (f18 > 93.5) & (f20 > 232.5)

In [248]:
import sympy
sympy.reduce_inequalities(inequalities_combinations[0][1])

False

In [301]:
measured_combinations_3

[(((-oo < f2) & (-oo < f19) & (-oo < f17) & (-oo < f0) & (f0 <= 62.0) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 297.5) & (0.5 < f15) & (93.5 < f18) & (232.5 < f20) & (f15 < oo) & (f18 < oo),
   (-oo < f4) & (-oo < f20) & (f16 <= 214.0) & (f20 <= 92.5) & (f4 <= 0.5) & (0.5 < f2) & (157.5 < f16) & (f2 < oo)),
  False),
 (((-oo < f2) & (-oo < f19) & (-oo < f17) & (-oo < f0) & (f0 <= 62.0) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 297.5) & (0.5 < f15) & (93.5 < f18) & (232.5 < f20) & (f15 < oo) & (f18 < oo),
   (-oo < f5) & (-oo < f2) & (-oo < f16) & (-oo < f15) & (-oo < f1) & (f0 <= 81.5) & (f1 <= 0.5) & (f15 <= 0.5) & (f16 <= 145.5) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 277.5) & (f5 <= 0.5) & (15.5 < f17) & (54.5 < f19) & (63.0 < f0) & (93.5 < f18) & (232.5 < f20) & (f18 < oo)),
  False),
 (((-oo < f2) & (-oo < f19) & (-oo < f17) & (-oo < f0) & (f0 <= 62.0) & (f17 <= 21.5) & (f19 <= 83.5) & (f2 <= 0.5) & (f20 <= 297.5) & (0.5 < f15) & (93.5 <

In [316]:
list(rule_measurements.values())

[False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 

In [317]:
np.unique(list(rule_measurements.values()), return_counts=True)

(array([False,  True]), array([632085, 148540]))

In [306]:
for comb in rule_measurements:
    if comb[1] == True:
        print("yeh")

In [ ]:
def measure_rules(all_rules, n_jobs: int = 1):
    rule_combinations = list(combinations(all_rules, 2))
    print(f"Generated {len(rule_combinations)} combinations")

    with tqdm_joblib(tqdm(desc="measure_rules", total=len(rule_combinations))) as progress_bar:
        measured_combinations = Parallel(n_jobs=n_jobs, backend='loky')(
            delayed(
                lambda comb: (comb, rule_overlaps(comb[0], comb[1]))
            )(combination)
            for combination in rule_combinations
        )

    return dict(measured_combinations)

In [184]:
%%timeit
all_rule_measurements = measure_rules(all_rules, n_jobs=8)

Generated 780625 combinations


measure_rules:   6%|▌         | 47040/780625 [05:33<1:26:36, 141.17it/s]


KeyboardInterrupt: 

In [167]:
len(all_rules)

9371

In [122]:
len(all_rules)

9395